# SwarmSight - Species Calibration Tool

Generate custom convex hull priors for non-bee species.

**Usage:**
1. Load a video of your study species
2. Annotate 30-50 frames with tip positions
3. Export the custom priorPoints.csv
4. Use it in SwarmSight Web via the config JSON

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy scipy pandas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
from scipy.spatial import ConvexHull
import pandas as pd
from google.colab import drive, files

drive.mount('/content/drive')

# Set your video path here
VIDEO_PATH = '/content/drive/MyDrive/YourVideo.mp4'  # CHANGE THIS

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Loaded: {total_frames} frames')

In [ ]:
# Sample frames evenly across the video
N_FRAMES = 40
frame_indices = np.linspace(0, total_frames - 1, N_FRAMES, dtype=int)
frames = []

for idx in frame_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        frames.append((idx, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))

print(f'Sampled {len(frames)} frames')
cap.release()

In [ ]:
# Annotation tool
# Click on each frame to mark: left antenna tip, right antenna tip, proboscis tip
# Press 'n' for next frame, 's' to skip

annotations = []  # List of {frame, region, x, y}
current_region = 'left_antenna'
regions = ['left_antenna', 'right_antenna', 'proboscis']
region_idx = 0

for frame_idx, frame_img in frames:
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(frame_img)
    ax.set_title(f'Frame {frame_idx} - Click: {regions[region_idx]} tip (3 clicks per frame)')
    
    points = plt.ginput(3, timeout=0)
    plt.close()
    
    if len(points) == 3:
        for i, (x, y) in enumerate(points):
            annotations.append({
                'frame': frame_idx,
                'region': regions[i],
                'x': x / frame_img.shape[1],  # Normalize to 0-1
                'y': y / frame_img.shape[0],
            })
        print(f'  Frame {frame_idx}: annotated')
    else:
        print(f'  Frame {frame_idx}: skipped')

print(f'\nTotal annotations: {len(annotations)}')

In [ ]:
# Compute convex hulls from annotations
df = pd.DataFrame(annotations)

hulls = {}
for region in regions:
    region_df = df[df['region'] == region]
    if len(region_df) >= 3:
        points = region_df[['x', 'y']].values
        hull = ConvexHull(points)
        hull_points = points[hull.vertices]
        hulls[region] = hull_points.tolist()
        print(f'{region}: {len(hull_points)} hull vertices from {len(region_df)} annotations')
    else:
        print(f'{region}: not enough annotations (need >= 3, got {len(region_df)})')

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {'left_antenna': '#22c55e', 'right_antenna': '#ef4444', 'proboscis': '#3b82f6'}

for ax, region in zip(axes, regions):
    region_df = df[df['region'] == region]
    ax.scatter(region_df['x'], region_df['y'], c=colors[region], s=20, alpha=0.6)
    if region in hulls:
        hull_pts = np.array(hulls[region] + [hulls[region][0]])
        ax.plot(hull_pts[:, 0], hull_pts[:, 1], c=colors[region], linewidth=2)
    ax.set_title(region)
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Export priorPoints.csv
df.to_csv('priorPoints.csv', index=False)
files.download('priorPoints.csv')
print('\nDownloaded priorPoints.csv')
print('Use this file in your SwarmSight Web config: set priors.customHullPath to this file.')